In [5]:
import re
import pandas as pd
import os

RATE_XLSX = r'C:\Users\prabh\Documents\Seatrium\CostAnalysis\Pipe_Rate_Table\Pipe_Rate_Table.xlsx'
OUT_XLSX  = r'C:\Users\prabh\Documents\Seatrium\CostAnalysis\Pipe_Rate_Table\extracted_ZMM_POREPORT_All harmonized projects_new.xlsx'

folder = r"C:\Users\prabh\Documents\Seatrium\CostAnalysis\Pipe_Rate_Table"
file   = "ZMM_POREPORT_All harmonized projects_R1_20260513 4_with formulas.xlsx"
df     = pd.read_excel(os.path.join(folder, file), sheet_name="ZMM",skiprows=1)

# ---------------------------------------------------------------------------
# 1. Combine material descriptions 2..7
# ---------------------------------------------------------------------------
md_cols = [f'Material Description {i}' for i in range(2, 8)]
df['Combined Material Description'] = df[md_cols].apply(
    lambda r: ' '.join(str(v).strip() for v in r if pd.notna(v) and str(v).strip()),
    axis=1,
)

# ---------------------------------------------------------------------------
# 2. smart_label — mirrors CE formula (Code!E3:E17 → F col), first-match wins
#    Returns the pcode string from Code G column (e.g. "P01+P02")
#    ORDER must match the Code sheet row order exactly.
# ---------------------------------------------------------------------------
# (keyword, pcode_string)  ← same sequence as Code!E3:E17 / G3:G17
SOW_TABLE = [
    ("Installation & testing",    "P03+P04+P05"),
    ("Fab and Installation",       "P01+P02+P03+P04"),
    ("FABRICATION / INSTALLATION", "P01+P02+P03+P04"),
    ("FI & Test",                  "P00"),
    ("P00",                        "P00"),
    ("Installation",               "P03+P04"),
    ("Fabrication Only",           "P01+P02"),
    ("Fabrication",                "P01+P02"),
    ("Fab",                        "P01+P02"),
    ("P1+2",                       "P01+P02"),
    ("P01-P02",                    "P01+P02"),
    ("actual production",          "P01+P02"),
]

def smart_label(row):
    """
    Replicate CE (SOW) formula: search all description cols for the first
    keyword match in SOW_TABLE; return the pcode string (e.g. 'P01+P02').
    Falls back to Material Description 1 when nothing matches (same as old code).
    """
    # CE formula searches TEXTJOIN of ALL description columns (P..W = MD1..MD7)
    all_cols = ['Material Description 1'] + md_cols
    text = ' '.join(
        str(row[c]).strip()
        for c in all_cols
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip()
    )
    t = text.upper()
    for keyword, pcode_str in SOW_TABLE:
        if keyword.upper() in t:
            return pcode_str          # ← returns "P01+P02", "P03+P04", etc.
    return str(row.get('Material Description 1', '')).strip()

mask = df['Order Unit'] == 'DI'
df["CHANGED DESCRIPTION"] = df.loc[mask].apply(smart_label, axis=1)

# ---------------------------------------------------------------------------
# 3. Regex helpers  (unchanged from original)
# ---------------------------------------------------------------------------
MAT_RE = re.compile(
    r'^\s*P\s*[-/]\s*'
    r'(?P<letters>[A-Z]{2,6})'
    r'(?P<digits>\d{0,4})\b',
    re.I,
)
SIZE_RE     = re.compile(r'(<\s*3\s*"?|>\s*3\s*"?|3\s*"?|<\s*3\b|>\s*3\b)', re.I)
LOC_RE      = re.compile(r'(?:\(\s*([A-D])\s*\)|/([A-D])/)', re.I)
LOC_CODE_RE = re.compile(r'\b([A-Z]\d{2})\b')
PCODE_RE    = re.compile(r'\b(?:P\d{2}|P/[SCDL])\b', re.I)
RANGE_RE    = re.compile(r'\b(P)(\d{2})\s*[-–]\s*(P)(\d{2})\b')
QTY_UNIT_RE = re.compile(r'-\s*(\d+(?:\.\d+)?)\s*(DI|M|KG|PIECE|RACK|METER|EA|PCS|PC)\b', re.I)

VALID_PCODES_RE = re.compile(
    r'\b(?:P00|P01|P02|P03|P04|P05|P1\+2|P\d{2}-P\d{2}|P/[SCDL])\b',
    re.I,
)
HAS_PCODE_RE = re.compile(r'P\s*[-/]\s*[A-Z]{2,6}\d{0,4}', re.I)


def normalize_size(tok):
    if tok is None:
        return None
    t = re.sub(r'\s+', '', tok)
    return f'{"<" if t.startswith("<") else ">"}3"'


def find_size(*texts):
    for t in texts:
        if t is None or pd.isna(t):
            continue
        m = SIZE_RE.search(str(t))
        if m:
            return normalize_size(m.group(1))
    return None


def expand_pcodes(s):
    """
    Expand ranges (P01-P02 → P01, P02) and split '+'-joined codes
    (P01+P02+P03+P04 → [P01, P02, P03, P04]).
    Normalise P1+2 → P01+P02 first.
    """
    s = re.sub(r'\bP1\+2\b', 'P01+P02', s, flags=re.I)
    # split on '+' so "P01+P02+P03+P04" is treated as separate tokens
    s = s.replace('+', ' ')
    se = s
    for L1, n1, L2, n2 in RANGE_RE.findall(s):
        if L1 == L2:
            a, b = int(n1), int(n2)
            if 0 <= a <= b <= 99:
                exp = ' '.join(f'{L1}{i:02d}' for i in range(a, b + 1))
                se = re.sub(rf'\b{L1}{n1}\s*[-–]\s*{L2}{n2}\b', exp, se)
    pcs = PCODE_RE.findall(se)
    pcs = [pu for p in pcs if VALID_PCODES_RE.fullmatch(pu := p.upper())]
    seen = set()
    return [p for p in pcs if not (p in seen or seen.add(p))]


def extract_loc_code(combined):
    if pd.isna(combined) or not str(combined).strip():
        return None
    hits = LOC_CODE_RE.findall(str(combined))
    seen = set()
    unique = [h for h in hits if not (h in seen or seen.add(h))]
    return ', '.join(unique) if unique else None


def extract_md1(cd, md1, combined):
    if pd.isna(md1):
        return False, None, None, None, None, None, []

    s  = str(md1).strip()
    cd = str(cd).strip() if not pd.isna(cd) else ""

    if not (VALID_PCODES_RE.search(s) or HAS_PCODE_RE.search(s)):
        return False, None, None, None, None, None, []

    mm      = MAT_RE.search(s)
    letters = mm.group('letters').upper() if mm else None
    digits  = mm.group('digits')          if mm else None
    if digits == '':
        digits = None

    if letters:
        material_map = {
            "DU":     "DSS",
            "DUPLEX": "DSS",
            "DSS":    "DSS",
        }
        letters = material_map.get(letters.upper(), letters.upper())

    size_lit = find_size(s, combined)
    bore     = None
    if size_lit:
        bore = 'SMALL' if size_lit.startswith('<') else 'LARGE'

    lm  = LOC_RE.search(s)
    loc = None
    if lm:
        loc = (lm.group(1) or lm.group(2)).upper()

    # ── pcode expansion: prefer CHANGED DESCRIPTION, fall back to md1 ──────
    pcode_source = cd if cd else s
    pcs = expand_pcodes(pcode_source)
    if not pcs:
        pcs = expand_pcodes(s)

    pcs = [pu for p in pcs if VALID_PCODES_RE.fullmatch(pu := p.upper())]
    if not pcs:
        return False, None, None, None, None, None, []

    full_key = f'{letters}{digits}' if (letters and digits) else letters
    if bore is None and full_key in {
        'CS0040', 'CS0120', 'CS0160', 'CS4080',
        'SS0040', 'SS0080', 'SS0160', 'SS1020',
        'CUNI', 'DU0040', 'DU0080', 'DU0160', 'GRE', 'S0080',
        'SDSS0040', 'SDSS0080', 'SDSS0160',
    }:
        bore = 'LARGE'

    return True, letters, digits, size_lit, bore, loc, pcs


def extract_qty(s):
    if pd.isna(s) or not str(s).strip():
        return None, None
    m = QTY_UNIT_RE.search(str(s))
    return (float(m.group(1)), m.group(2).upper()) if m else (None, None)


# ---------------------------------------------------------------------------
# 4. Rate lookup  (unchanged from original)
# ---------------------------------------------------------------------------
ALIAS = {
    'CS040':  'CS0040',
    'CS8160': 'CS0160',
    'CSX65':  'CS0160',
    'CSX52':  'CS0160',
    'SS':     'SS0040',
    'SS0080': 'S0080',
}
CS_FAMILY = {'CS0040', 'CS0120', 'CS0160', 'CS4080'}


def normalize_material(letters, digits):
    if letters is None:
        return None
    raw = f'{letters}{digits}' if digits else letters
    return ALIAS.get(raw, raw)


rate     = pd.read_excel(RATE_XLSX, sheet_name='Worksheet')
rate_idx = {}
for _, r in rate.iterrows():
    rate_idx[(r['MaterialKey'], r['Scope Code'], r['BoreType'])] = {
        'A': r['A'], 'B': r['B'], 'C': r['C'], 'D': r['D'],
    }


def lookup_rate(letters, digits, pcode, bore, location):
    if not letters or not pcode or not location:
        return None
    mat  = normalize_material(letters, digits)
    bore = bore if bore in ('LARGE', 'SMALL') else 'LARGE'
    if mat in CS_FAMILY and bore == 'SMALL':
        row = rate_idx.get((mat, pcode, 'LARGE'))
        if row is None:
            return None
        v = row.get(location)
        return None if v is None or pd.isna(v) else round(v * 1.20, 4)
    row = rate_idx.get((mat, pcode, bore)) or rate_idx.get((mat, pcode, 'LARGE'))
    if row is None:
        return None
    v = row.get(location)
    return None if v is None or pd.isna(v) else v


# ---------------------------------------------------------------------------
# 5. Build output
# ---------------------------------------------------------------------------
records = []
for _, r in df.iterrows():
    combined     = r['Combined Material Description']
    pcode_source = r['CHANGED DESCRIPTION']

    is_p, letters, digits, size, bore, loc, pcs = extract_md1(
        pcode_source, r['Material Description 1'], combined
    )

    # ── Qty: regex first, then fall back to source columns ──────────────────
    qty, unit = extract_qty(combined)
    if qty is None:
        src_qty  = r.get('Order Quantity')
        src_unit = r.get('Order Unit')
        if pd.notna(src_qty) and src_qty not in (None, ''):
            try:
                qty  = float(src_qty)
                unit = str(src_unit).strip() if pd.notna(src_unit) else None
            except (ValueError, TypeError):
                pass

    loc_code = extract_loc_code(combined)

    # ── Predicted cost ───────────────────────────────────────────────────────
    pred = None
    if is_p and qty and pcs:
        rs = [lookup_rate(letters, digits, pc, bore, loc) for pc in pcs]
        rs = [x for x in rs if x is not None and not pd.isna(x)]
        if rs:
            pred = round(sum(rs) * qty, 2)

    # ── If predicted still None, fall back to Actual / Net Order Value ───────
    actual = r.get('Net Order Value - Project Rate')
    if pred is None and pd.notna(actual):
        pred = actual

    records.append({
        'WBS element':                   r.get('WBS element'),
        'WBS Description':               r.get('WBS Element Description'),
        'PO Order':                      r.get('Purchasing Document'),
        'Item':                          r.get('Item'),
        'Material Description 1':        r.get('Material Description 1'),
        'CHANGED DESCRIPTION':           r.get('CHANGED DESCRIPTION'),
        'Combined Material Description': combined,
        'Is Pcode':                      is_p,
        'Extracted Pcode':               ', '.join(pcs) if pcs else None,
        'Extracted Material':            letters,
        'Extracted Schedule':            digits,
        'Extracted Size':                size,
        'Extracted Location':            loc,
        'Extracted Location Code':       loc_code,
        'Extracted Qty':                 qty,
        'Extracted Unit':                unit,
        'Unit (Source)':                 r.get('Order Unit'),
        'Order Quantity (Source)':       r.get('Order Quantity'),
        'Predicted Cost':                pred,
        'Actual Cost':                   actual,
        'Currency':                      r.get('Currency'),
        'Project':                       r.get('Project'),
        'Delivery date':                 r.get('Delivery date'),
        'Created On':                    r.get('Created On'),
    })

out = pd.DataFrame.from_records(records)

# ---------------------------------------------------------------------------
# 6. Save
# ---------------------------------------------------------------------------
with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as xw:
    out.to_excel(xw, index=False, sheet_name='Extracted')

print(f'Total rows           : {len(out)}')
print(f'  Pcode rows         : {out["Is Pcode"].sum()}')
print(f'  Predicted cost done: {out["Predicted Cost"].notna().sum()}')
print(f'  Material extracted : {out["Extracted Material"].notna().sum()}')
print(f'  Schedule extracted : {out["Extracted Schedule"].notna().sum()}')
print(f'  Size extracted     : {out["Extracted Size"].notna().sum()}')
print(f'  Location code found: {out["Extracted Location Code"].notna().sum()}')
print(f'\nSaved: {OUT_XLSX}')


Total rows           : 88619
  Pcode rows         : 283
  Predicted cost done: 88619
  Material extracted : 187
  Schedule extracted : 183
  Size extracted     : 264
  Location code found: 35767

Saved: C:\Users\prabh\Documents\Seatrium\CostAnalysis\Pipe_Rate_Table\extracted_ZMM_POREPORT_All harmonized projects_new.xlsx


In [3]:
print(df.columns)

Index([      'Unnamed: 0',       'Unnamed: 1',       'Unnamed: 2',
             'Unnamed: 3',       'Unnamed: 4',       'Unnamed: 5',
             'Unnamed: 6',       'Unnamed: 7',       'Unnamed: 8',
             'Unnamed: 9',      'Unnamed: 10',      'Unnamed: 11',
            'Unnamed: 12',      'Unnamed: 13',      'Unnamed: 14',
            'Unnamed: 15',      'Unnamed: 16',      'Unnamed: 17',
            'Unnamed: 18',      'Unnamed: 19',      'Unnamed: 20',
            'Unnamed: 21',      'Unnamed: 22',      'Unnamed: 23',
            'Unnamed: 24',      'Unnamed: 25',      'Unnamed: 26',
            'Unnamed: 27',      'Unnamed: 28',      'Unnamed: 29',
            'Unnamed: 30',      'Unnamed: 31', 15446090135.250246,
            'Unnamed: 33',  15417910568.66034,      'Unnamed: 35',
            'Unnamed: 36',      'Unnamed: 37',      'Unnamed: 38',
            'Unnamed: 39',      'Unnamed: 40',      'Unnamed: 41',
            'Unnamed: 42',      'Unnamed: 43',      'Unnamed: 

In [18]:
import re
import pandas as pd
import os
RATE_XLSX = r'C:\Users\Prabha.Sivasankar-C\OneDrive - Seatrium Ltd\Documents\CostAnlaysis\Jun2026\Pipe_Rate_Table.xlsx'
OUT_XLSX  = r'C:\Users\Prabha.Sivasankar-C\OneDrive - Seatrium Ltd\Documents\CostAnlaysis\Jun2026\extracted_ZMM_POREPORT_All harmonized projects_new.xlsx'

folder = r"C:\Users\Prabha.Sivasankar-C\OneDrive - Seatrium Ltd\Documents\CostAnlaysis\Jun2026"
file   = "ZMM_POREPORT_All harmonized projects_R1_20260513 - testing.xlsx"
df     = pd.read_excel(os.path.join(folder, file), sheet_name="Sheet2")

# ---------------------------------------------------------------------------
# 1. Combine material descriptions 2..7
# ---------------------------------------------------------------------------
md_cols = [f'Material Description {i}' for i in range(2, 8)]
df['Combined Material Description'] = df[md_cols].apply(
    lambda r: ' '.join(str(v).strip() for v in r if pd.notna(v) and str(v).strip()),
    axis=1,
)

# ---------------------------------------------------------------------------
# 2. smart_label — mirrors CE formula (Code!E3:E17 → G col pcode string)
#    Searches ALL description columns (MD1..MD7), first-match wins.
# ---------------------------------------------------------------------------
SOW_TABLE = [
    ("Installation & testing",    "P03+P04+P05"),
    ("INS/TESTING",    "P03+P04+P05"),
    ("INS /TESTING",    "P03+P04+P05"),    
    ("Fab and Installation",       "P01+P02+P03+P04"),
    ("FABRICATION / INSTALLATION", "P01+P02+P03+P04"),
    ("FI & Test",                  "P00"),
    ("P00",                        "P00"),
    ("Installation",               "P03+P04"),
    ("Fabrication Only",           "P01+P02"),
    ("Fabrication",                "P01+P02"),
    ("Fab",                        "P01+P02"),
    ("P1+2",                       "P01+P02"),
    ("P01-P02",                    "P01+P02"),
    ("actual production",          "P01+P02"),
]

def smart_label(row):
    all_cols = ['Material Description 1'] + md_cols
    text = ' '.join(
        str(row[c]).strip()
        for c in all_cols
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip()
    )
    t = text.upper()
    for keyword, pcode_str in SOW_TABLE:
        if keyword.upper() in t:
            return pcode_str
    return str(row.get('Material Description 1', '')).strip()

mask = df['Order Unit'] == 'DI'
df["CHANGED DESCRIPTION"] = df.loc[mask].apply(smart_label, axis=1)

# ---------------------------------------------------------------------------
# 3. Regex helpers
# ---------------------------------------------------------------------------
# ── Material extraction from free text (CD formula logic) ──────────────────
# Priority order matches Excel CD formula exactly (SDSS must come before SS)
MAT_KEYWORDS = [
     ("DUPLEX", "DSS"),
    ("/DU",    "DSS"),
    ("P/CS",   "CS"),
    ("CUNI",   "CUNI"),
    ("LTCS",   "LTCS"),
    ("CS",     "CS"),
    ("CPVC",   "CPVC"),
    ("SDSS",   "SDSS"),
    ("SS",     "SS"),      # only if SDSS not present (handled in function)
    ("DSS",    "DSS"),
    ("-DUPLEX", "DSS"),
    ("-CUNI",   "CUNI"),
    ("-LTCS",   "LTCS"),
    ("-CS",     "CS"),
    ("-CPVC",   "CPVC"),
    ("-SDSS",   "SDSS"),
    ("-SS",     "SS"),      # only if SDSS not present (handled in function)
    ("-DSS",    "DSS"),    
]

SCH_MAP_Keywords = {    
    '1.781" / 45.2MM': '140',
    '2.25" / 57.1MM': '160',
    '2" / 50.8MM': '160',
    '2"/50': '160',
    'XXS': 'XXS',
}



def extract_material_from_text(text):
    """Extract material type from any free text (mirrors CD formula logic)."""
    if not text or pd.isna(text):
        return None
    t = text.upper()
    # print(t,"\n")
    has_sdss = "SDSS" in t
    for keyword, result in MAT_KEYWORDS:
        kw = keyword.upper()
        if kw == "SS" and has_sdss:
            continue   # guard: skip SS match when SDSS present
        if kw in t:
            # print("\n",result,"\n")
            return result
    return None

# Material map for letter-based extraction (from P/SS0040/... style codes)
MATERIAL_MAP = {
    "DU":     "DSS",
    "DUPLEX": "DSS",
    "DSS":    "DSS",
}

# MAT_RE      = re.compile(r'^\s*P\s*[-/]\s*(?P<letters>[A-Z]{2,6})(?P<digits>\d{0,4})\b', re.I)
MAT_RE = re.compile(
    r'(?:\b|/|-)(?P<letters>SDSS|DSS|SS|DUPLEX|CUNI|LTCS|CPVC|CS|DU)(?P<digits>\d{0,4})\b',
    re.I
)
DIGIT_RE = re.compile(
    r'/(?:SDSS|DSS|SS|DUPLEX|CUNI|LTCS|CPVC|CS|DU)(\d+)'
    r'|SCH(?:EDULE)?\s*(\d+|XXS|XS|STD)',
    re.I
)


SIZE_RE     = re.compile(r'(<\s*3\s*"?|>\s*3\s*"?|3\s*"?|<\s*3\b|>\s*3\b)', re.I)
LOC_RE      = re.compile(r'(?:\(\s*([A-D])\s*\)|/([A-D])/)', re.I)
LOC_CODE_RE = re.compile(r'\b([A-Z]\d{2})\b')
PCODE_RE    = re.compile(r'\b(?:P\d{2}|P/[SCDL])\b', re.I)
RANGE_RE    = re.compile(r'\b(P)(\d{2})\s*[-–]\s*(P)(\d{2})\b')
QTY_UNIT_RE = re.compile(r'-\s*(\d+(?:\.\d+)?)\s*(DI|M|KG|PIECE|RACK|METER|EA|PCS|PC)\b', re.I)

VALID_PCODES_RE = re.compile(
    r'\b(?:P00|P01|P02|P03|P04|P05|P1\+2|P\d{2}-P\d{2}|P/[SCDL])\b', re.I
)
HAS_PCODE_RE = re.compile(r'P\s*[-/]\s*[A-Z]{2,6}\d{0,4}', re.I)

# Size keywords in plain text (from CG formula + common variants)
SIZE_PLAIN_RE = re.compile(
    r'(less\s+than\s+3|below\s+3|<\s*3|>=\s*3|>\s*3|3"\s*and\s*above|3"\s*and\s*below'
    r'|>=3|<=3|<3|>3)',
    re.I
)

def normalize_size(tok):
    if tok is None:
        return None
    t = re.sub(r'\s+', '', tok.lower())
    if any(x in t for x in ['<3', 'below3', 'lessthan3', '<=3']):
        return '<3"'
    return '>3"'

def find_size(*texts):
    """
    Look for size in any text — first tries the original SIZE_RE (for P/SS0040/<3"/...),
    then plain-English patterns (LESS THAN 3", >=3", etc.).
    """
    for t in texts:
        if t is None or pd.isna(t):
            continue
        s = str(t)
        m = SIZE_RE.search(s)
        if m:
            raw = re.sub(r'\s+', '', m.group(1))
            return '<3"' if raw.startswith('<') else '>3"'
        m2 = SIZE_PLAIN_RE.search(s)
        if m2:
            return normalize_size(m2.group(1))
    return None

def expand_pcodes(s):
    """Expand ranges and '+'-joined pcode strings into a list."""
    s = re.sub(r'\bP1\+2\b', 'P01+P02', s, flags=re.I)
    s = s.replace('+', ' ')
    se = s
    for L1, n1, L2, n2 in RANGE_RE.findall(s):
        if L1 == L2:
            a, b = int(n1), int(n2)
            if 0 <= a <= b <= 99:
                exp = ' '.join(f'{L1}{i:02d}' for i in range(a, b + 1))
                se = re.sub(rf'\b{L1}{n1}\s*[-–]\s*{L2}{n2}\b', exp, se)
    pcs = PCODE_RE.findall(se)
    pcs = [pu for p in pcs if VALID_PCODES_RE.fullmatch(pu := p.upper())]
    seen = set()
    return [p for p in pcs if not (p in seen or seen.add(p))]

def extract_loc_code(combined):
    if pd.isna(combined) or not str(combined).strip():
        return None
    hits = LOC_CODE_RE.findall(str(combined))
    seen = set()
    unique = [h for h in hits if not (h in seen or seen.add(h))]
    return ', '.join(unique) if unique else None

def extract_qty(s):
    if pd.isna(s) or not str(s).strip():
        return None, None
    m = QTY_UNIT_RE.search(str(s))
    return (float(m.group(1)), m.group(2).upper()) if m else (None, None)

# ---------------------------------------------------------------------------
# 4. Main extraction — handles BOTH styles:
#    A) Classic: MD1 = "P/SS0040/<3"/C/P01-P02/24"   (pcode in MD1)
#    B) New:     MD1 = "PIPING WORKER (2024)"          (pcode in CHANGED DESC)
# ---------------------------------------------------------------------------
def extract_row(cd, md1, combined,material):
    """
    Returns (is_p, letters, digits, size, bore, loc, pcs)

    Strategy:
    1. Try classic regex on MD1 (style A).
    2. If that fails, try CHANGED DESCRIPTION for pcodes (style B).
       For style B, extract material + size from Combined Material Description.
    """
    cd  = str(cd).strip()  if not pd.isna(cd)  else ""
    md1 = str(md1).strip() if not pd.isna(md1) else ""
    material_text = "" if pd.isna(material) else str(material)
    # ── Try classic style A first ───────────────────────────────────────────
    letters, digits, size_lit, bore, loc = None, None, None, None, None
    
    mm_digit=DIGIT_RE.search(md1 or "") or DIGIT_RE.search(combined)or DIGIT_RE.search(material_text)   
    if mm_digit:
        digits = mm_digit.group(1) or mm_digit.group(2)
        # digits = str(int(digits))  # remove leading zeros
        digits = "80" if digits == "8" else digits
    else:
        digits =SCH_MAP_Keywords.get(str(combined).upper(),"Without Schedule")
        # digits  = mm.group('digits') or None
        
    if VALID_PCODES_RE.search(md1) or HAS_PCODE_RE.search(md1) or VALID_PCODES_RE.search(material_text) or HAS_PCODE_RE.search(material_text):
        # mm = MAT_RE.search(md1)
        mm=  MAT_RE.search(md1 or "") or MAT_RE.search(combined)or MAT_RE.search(material_text)
        # mm_digit=DIGIT_RE.search(md1 or "") or DIGIT_RE.search(combined)or DIGIT_RE.search(material_text)
        if mm:
            letters = MATERIAL_MAP.get(mm.group('letters').upper(),
                                       mm.group('letters').upper())
            
        # if mm_digit:
        #     digits = mm_digit.group(1) or mm_digit.group(2)
        #     digits = str(int(digits))  # remove leading zeros
        #     digits = "80" if digits == "8" else digits
        # else:
        #     digits=None
        #     # digits  = mm.group('digits') or None

        size_lit = find_size(md1, combined)
        if size_lit:
            bore = 'SMALL' if size_lit.startswith('<') else 'LARGE'

        lm = LOC_RE.search(md1)
        if lm:
            loc = (lm.group(1) or lm.group(2)).upper()

        pcs = expand_pcodes(cd if cd else md1)
        if not pcs:
            pcs = expand_pcodes(md1)
        pcs = [pu for p in pcs if VALID_PCODES_RE.fullmatch(pu := p.upper())]

        if pcs:
            # bore fallback for known full-key combos
            full_key = f'{letters}{digits}' if (letters and digits) else letters
            if bore is None and full_key in {
                'CS0040','CS0120','CS0160','CS4080',
                'SS0040','SS0080','SS0160','SS1020',
                'CUNI','DU0040','DU0080','DU0160','GRE','S0080',
                'SDSS0040','SDSS0080','SDSS0160',
            }:
                bore = 'LARGE'
            return True, letters, digits, size_lit, bore, loc, pcs

    # ── Style B: pcode only in CHANGED DESCRIPTION ─────────────────────────
    if not cd:
        return False, None, None, None, None, None, []

    pcs = expand_pcodes(cd)
    pcs = [pu for p in pcs if VALID_PCODES_RE.fullmatch(pu := p.upper())]
    if not pcs:
        return False, None, None, None, None, None, []

    # Extract material from combined text (mirrors CD formula)
    search_text = (combined or '') + ' ' + md1
    # print(search_text)
    mat = extract_material_from_text(search_text)

    # Extract size from combined text (mirrors CG formula)
    size_lit = find_size(combined, md1)
    if size_lit:
        bore = 'SMALL' if size_lit.startswith('<') else 'LARGE'
    else:
        bore = 'LARGE'   # default

    # Extract location letter from combined text
    lm = LOC_RE.search(search_text)
    if lm:
        loc = (lm.group(1) or lm.group(2)).upper()

    # No schedule digits available for style B rows
    return True, mat, digits, size_lit, bore, loc, pcs

# ---------------------------------------------------------------------------
# 5. Rate lookup
# ---------------------------------------------------------------------------
ALIAS = {
    'CS040':  'CS0040',
    'CS8160': 'CS0160',
    'CSX65':  'CS0160',
    'CSX52':  'CS0160',
    'SS':     'SS0040',
    'SS0080': 'S0080',
}
CS_FAMILY = {'CS0040', 'CS0120', 'CS0160', 'CS4080'}

def normalize_material(letters, digits):
    if not letters:
        return None
    raw = f'{letters}{digits}' if digits else letters
    return ALIAS.get(raw, raw)

rate     = pd.read_excel(RATE_XLSX, sheet_name='Worksheet')
rate_idx = {}
for _, r in rate.iterrows():
    rate_idx[(r['MaterialKey'], r['Scope Code'], r['BoreType'])] = {
        'A': r['A'], 'B': r['B'], 'C': r['C'], 'D': r['D'],
    }

def lookup_rate(letters, digits, pcode, bore, location):
    if not letters or not pcode or not location:
        return None
    mat  = normalize_material(letters, digits)
    bore = bore if bore in ('LARGE', 'SMALL') else 'LARGE'
    if mat in CS_FAMILY and bore == 'SMALL':
        row = rate_idx.get((mat, pcode, 'LARGE'))
        if row is None:
            return None
        v = row.get(location)
        return None if v is None or pd.isna(v) else round(v * 1.20, 4)
    row = rate_idx.get((mat, pcode, bore)) or rate_idx.get((mat, pcode, 'LARGE'))
    if row is None:
        return None
    v = row.get(location)
    return None if v is None or pd.isna(v) else v

# ---------------------------------------------------------------------------
# 6. Build output
# ---------------------------------------------------------------------------
records = []
for _, r in df.iterrows():
    combined     = r['Combined Material Description']
    pcode_source = r.get('CHANGED DESCRIPTION', '')
    

    is_p, letters, digits, size, bore, loc, pcs = extract_row(
        pcode_source, r['Material Description 1'], combined,r['Material']
    )

    # Qty: regex first, then fall back to source columns
    qty, unit = extract_qty(combined)
    if qty is None:
        src_qty  = r.get('Order Quantity')
        src_unit = r.get('Order Unit')
        if pd.notna(src_qty) and src_qty not in (None, ''):
            try:
                qty  = float(src_qty)
                unit = str(src_unit).strip() if pd.notna(src_unit) else None
            except (ValueError, TypeError):
                pass

    loc_code = extract_loc_code(combined)

    # Predicted cost
    pred = None
    if is_p and qty and pcs:
        rs = [lookup_rate(letters, digits, pc, bore, loc) for pc in pcs]
        rs = [x for x in rs if x is not None and not pd.isna(x)]
        if rs:
            pred = round(sum(rs) * qty, 2)

    # Fallback: use actual cost when prediction unavailable
    actual = r.get('Net Order Value - Project Rate')
    if pred is None and pd.notna(actual):
        pred = actual

    records.append({
        'WBS element':                   r.get('WBS element'),
        'WBS Description':               r.get('WBS Element Description'),
        'PO Order':                      r.get('Purchasing Document'),
        'Item':                          r.get('Item'),
        'Material':r.get('Material'),
        'Material Description 1':        r.get('Material Description 1'),
        'CHANGED DESCRIPTION':           pcode_source,
        'Combined Material Description': combined,
        'Is Pcode':                      is_p,
        'Extracted Pcode':               ', '.join(pcs) if pcs else None,
        'Extracted Material':            letters,
        'Extracted Schedule':            digits,
        'Extracted Size':                size,
        'Extracted Location':            loc,
        'Extracted Location Code':       loc_code,
        'Extracted Qty':                 qty,
        'Extracted Unit':                unit,
        'Unit (Source)':                 r.get('Order Unit'),
        'Order Quantity (Source)':       r.get('Order Quantity'),
        'Predicted Cost':                pred,
        'Actual Cost':                   actual,
        'Currency':                      r.get('Currency'),
        'Project':                       r.get('Project'),
        'Delivery date':                 r.get('Delivery date'),
        'Created On':                    r.get('Created On'),
    })

out = pd.DataFrame.from_records(records)

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as xw:
    out.to_excel(xw, index=False, sheet_name='Extracted')

print(f'Total rows           : {len(out)}')
print(f'  Pcode rows         : {out["Is Pcode"].sum()}')
print(f'  Predicted cost done: {out["Predicted Cost"].notna().sum()}')
print(f'  Material extracted : {out["Extracted Material"].notna().sum()}')
print(f'  Schedule extracted : {out["Extracted Schedule"].notna().sum()}')
print(f'  Size extracted     : {out["Extracted Size"].notna().sum()}')
print(f'  Location found     : {out["Extracted Location"].notna().sum()}')
print(f'  Location code found: {out["Extracted Location Code"].notna().sum()}')
print(f'\nSaved: {OUT_XLSX}')

Total rows           : 70610
  Pcode rows         : 440
  Predicted cost done: 70610
  Material extracted : 440
  Schedule extracted : 440
  Size extracted     : 395
  Location found     : 187
  Location code found: 32642

Saved: C:\Users\Prabha.Sivasankar-C\OneDrive - Seatrium Ltd\Documents\CostAnlaysis\Jun2026\extracted_ZMM_POREPORT_All harmonized projects_new.xlsx
